# Sleep Planner

In [ ]:
import math
from datetime import datetime, timedelta

In [ ]:
COFFEES = [
    ("กาแฟดำชงผ่านกระดาษ (Drip)", 240, 95, 96),
    ("กาแฟดำ Drip แก้วใหญ่", 350, 140, 140),
    ("Espresso 1 Shot", 30, 63, 63),
    ("Espresso 2 Shots", 60, 126, 126),
    ("Americano 1 Shot", 240, 63, 63),
    ("Americano 2 Shots", 350, 126, 126),
    ("Latte 1 Shot", 240, 63, 63),
    ("Cappuccino 1 Shot", 180, 63, 63),
    ("Cold Brew", 240, 100, 200),
    ("Instant Coffee", 240, 62, 62),
    ("Decaf Coffee", 240, 1, 5),
]

TARGETS = [
    (100, "เหลือไม่เกิน 100 mg"),
    (50, "เหลือไม่เกิน 50 mg"),
    (1, "ใกล้หมด (ต่ำกว่า 1 mg)")
]

In [ ]:
def convert_time(value):
    value = value.strip()

    # รองรับ 12:00, 1200, 12000, 830
    if ":" in value:
        parts = value.split(":")
        if len(parts) != 2:
            raise ValueError("รูปแบบเวลาไม่ถูกต้อง")
        hour, minute = parts
    else:
        if len(value) == 5 and value.endswith("0"):
            value = value[:-1]

        if not value.isdigit():
            raise ValueError("กรุณากรอกเวลาเป็นตัวเลข")

        if len(value) <= 2:
            hour, minute = value, "00"
        elif len(value) in (3, 4):
            hour, minute = value[:-2], value[-2:]
        else:
            raise ValueError("รูปแบบเวลาไม่ถูกต้อง")

    hour = int(hour)
    minute = int(minute)

    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        raise ValueError("ชั่วโมงหรือนาทีไม่ถูกต้อง")

    return datetime.strptime(f"{hour:02d}:{minute:02d}", "%H:%M")

In [ ]:
def calc_vd(weight):

    return weight * 0.6


def calc_c0(dose, vd):
    """ความเข้มข้นเริ่มต้น C0 (สมมติว่าคาเฟอีนเข้าสู่ร่างกายทันที)"""
    return dose / vd


def calc_cl(weight, age):

    cl_per_kg = 0.06 if age >= 65 else 0.085
    return cl_per_kg * weight


def calc_half_life(vd, cl):

    return math.log(2) * vd / cl


def calc_ke(t_half):

    return math.log(2) / t_half


def calc_pk(weight, age, dose):

    vd = calc_vd(weight)
    cl = calc_cl(weight, age)
    t_half = calc_half_life(vd, cl)
    return {
        "Vd": vd,
        "C0": calc_c0(dose, vd),
        "CL": cl,
        "t_half": t_half,
        "ke": calc_ke(t_half),
    }


def minutes_to_target(dose, ke, target):

    hours = math.log(dose / target) / ke
    if target == 1:
        # ปัดไปนาทีถัดไป เพื่อให้ต่ำกว่า 1 mg จริง
        return max(0, math.floor(hours * 60) + 1)
    return max(0, math.ceil(hours * 60))


def calc_alerts(start_time, dose, ke, targets=TARGETS):
    
    alerts = []
    for target, label in targets:
        minutes = minutes_to_target(dose, ke, target)
        end_time = start_time + timedelta(minutes=minutes)
        alerts.append({
            "label": label,
            "minutes": minutes,
            "end_time": end_time,
            "days_later": (end_time.date() - start_time.date()).days,
        })
    return alerts

In [ ]:
# ======================================
# แบบจำลองหลายแก้ว (Superposition)
# A(t) = Σ Dᵢ · e^(−ke (t − tᵢ))   สำหรับแก้วที่ tᵢ ≤ t
# เวลา = ชั่วโมงนับจาก 00:00 ของวันที่ดื่ม (≥ 24 = วันถัดไป)
# ======================================

SLEEP_MG = 50    # เกณฑ์หลัก (Gardiner et al., 2024)
STRICT_MG = 30   # เกณฑ์เข้มงวด (Gardiner et al., 2023 meta-analysis)


def caffeine_at(t, doses, ke):
    """คาเฟอีนรวม (mg) ณ เวลา t จากทุกแก้ว doses = [(ชั่วโมง, mg), ...]"""
    return sum(mg * math.exp(-ke * (t - hour)) for hour, mg in doses if hour <= t)


def find_sleep_time(doses, ke, threshold):
    """เวลาแรก (ปัดขึ้นเป็นนาที) หลังแก้วสุดท้ายที่คาเฟอีน ≤ threshold
    คืน None ถ้าต่ำกว่าเกณฑ์ตั้งแต่หลังแก้วสุดท้ายแล้ว"""
    t_last = max(hour for hour, _ in doses)
    amount = caffeine_at(t_last, doses, ke)
    if amount <= threshold:
        return None
    # หลังแก้วสุดท้าย A(t) = A(t_last)·e^(−ke·Δt)  →  Δt = ln(A/threshold)/ke
    hours = t_last + math.log(amount / threshold) / ke
    return math.ceil(round(hours * 60, 6)) / 60


def hours_to_label(h):
    """28.5 → '04:30 (วันถัดไป)'"""
    total_minutes = round(h * 60)
    days, minutes = divmod(total_minutes, 24 * 60)
    return f"{minutes // 60:02d}:{minutes % 60:02d} ({day_label(days)})"

## หัวข้อคณิตศาสตร์ที่ 1: Sequence & Series (Lec7, Lab4)

**ลำดับเรขาคณิต** — กาแฟ 1 แก้ว ปริมาณ $D$ เมื่อผ่านไป $n$ ครึ่งชีวิต จะเหลือ

$$a_n = D \cdot \left(\tfrac{1}{2}\right)^n \qquad (r = \tfrac12)$$

**อนุกรมเรขาคณิต** — ช่วงสอบดื่มแบบเดิมทุกวัน ให้ $B$ = คาเฟอีนตอนเข้านอนจากวันเดียว และ $r = e^{-24 k_e}$ คือสัดส่วนที่เหลือหลังผ่านไป 1 วัน
คาเฟอีนตอนนอนของวันที่ $n$ คือผลรวมของสิ่งที่ค้างจากทุกวันก่อนหน้า:

$$S_n = B + Br + Br^2 + \dots + Br^{n-1} = B\,\frac{1 - r^n}{1 - r}$$

เพราะ $|r| < 1$ อนุกรมนี้ **ลู่เข้า (convergent)** สู่

$$S_\infty = \frac{B}{1 - r}$$

ในโปรแกรมคำนวณ $S_n$ สองวิธี — ผลรวมสะสมด้วย `for` loop (partial sums แบบ Lab4) และสูตรปิด — แล้วเทียบว่าตรงกัน

ข้อสังเกต: ผู้ใหญ่ทั่วไป $t_{1/2} \approx 4.9$ ชม. ดังนั้น 24 ชม. ≈ 5 ครึ่งชีวิต ทำให้ $r \approx 0.033$ คาเฟอีนจึงสะสมข้ามวันเพิ่มเพียงราว 3% ส่วนผู้สูงอายุ ($t_{1/2}$ ยาวกว่า) $r$ จะมากกว่าและสะสมมากกว่า

In [ ]:
# ======================================
# Sequence & Series (Lec7, Lab4)
# ======================================

def half_life_sequence(dose, t_half, n_max=6):
    """ลำดับเรขาคณิต aₙ = D·(½)ⁿ  เมื่อผ่านไป n ครึ่งชีวิต
    คืน [(n, ชั่วโมงหลังดื่ม, mg), ...]"""
    return [(n, n * t_half, dose * 0.5 ** n) for n in range(n_max + 1)]


def multi_day_series(B, ke, days=7):
    """อนุกรมเรขาคณิต: ถ้าดื่มแบบเดิมทุกวัน คาเฟอีนตอนนอนวันที่ n
    Sₙ = B + B·r + … + B·r^(n−1) = B(1 − rⁿ)/(1 − r),  r = e^(−24·ke)
    ลู่เข้าสู่ S∞ = B/(1 − r) เพราะ |r| < 1"""
    r = math.exp(-24 * ke)
    rows = []
    loop_sum = 0.0
    for n in range(1, days + 1):
        loop_sum += B * r ** (n - 1)              # ผลรวมสะสม (partial sum) แบบ Lab4
        formula_sum = B * (1 - r ** n) / (1 - r)  # สูตรปิดของอนุกรมเรขาคณิต
        rows.append((n, loop_sum, formula_sum))
    return {"r": r, "rows": rows, "limit": B / (1 - r)}


def show_series(cups, ke, t_half, bed_hour):
    name, _, dose, _, _ = max(cups, key=lambda cup: cup[2])
    print(f"\n[ลำดับเรขาคณิต] {name} {dose:.0f} mg — เหลือครึ่งหนึ่งทุก {t_half:.2f} ชม.")
    print(f"{'n':>3} | {'หลังดื่ม (ชม.)':>14} | {'เหลือ (mg)':>10}")
    for n, hours, mg in half_life_sequence(dose, t_half):
        print(f"{n:>3} | {hours:>14.2f} | {mg:>10.2f}")

    B = caffeine_at(bed_hour, to_doses(cups), ke)
    s = multi_day_series(B, ke)
    print(f"\n[อนุกรมเรขาคณิต] ถ้าดื่มแบบนี้ทุกวันช่วงสอบ — คาเฟอีนตอนนอน {hours_to_label(bed_hour)}")
    print(f"B = {B:.2f} mg, r = e^(−24·ke) = {s['r']:.4f}  (|r| < 1 → ลู่เข้า)")
    print(f"{'วันที่':>5} | {'ผลรวมสะสม (loop)':>16} | {'สูตร B(1−rⁿ)/(1−r)':>18}")
    for n, loop_sum, formula_sum in s["rows"]:
        print(f"{n:>5} | {loop_sum:>16.4f} | {formula_sum:>18.4f}")
    print(f"ลิมิต S∞ = B/(1−r) = {s['limit']:.2f} mg")
    if B > 0:
        print(f"สรุป: สะสมเพิ่มจากวันแรกเพียง {(s['limit'] / B - 1) * 100:.1f}% แล้วคงที่ ไม่พุ่งขึ้นเรื่อย ๆ")

In [ ]:
# ======================================
# กราฟ (Lab5: matplotlib) — ข้อความในกราฟเป็นภาษาอังกฤษเพื่อเลี่ยงปัญหาฟอนต์ไทย
# ======================================
import numpy as np
import matplotlib.pyplot as plt

TOTAL_COLOR = "#f59e0b"
OK_COLOR = "#16a34a"
CUP_COLORS = ["#60a5fa", "#a78bfa", "#f472b6", "#34d399", "#fbbf24",
              "#f87171", "#22d3ee", "#c084fc", "#fb923c", "#94a3b8"]


def _time_axis(doses, sleep_hour=None):
    """ช่วงเวลาในกราฟ: ก่อนแก้วแรก 1 ชม. ถึงอย่างน้อย 24 ชม. (หรือหลังเวลานอนได้)"""
    first = min(hour for hour, _ in doses)
    start = first - 1
    end = max(sleep_hour or 0, first + 24) + 1
    return np.linspace(start, end, 1000), start, end


def _format_hours(ax, start, end):
    ticks = np.arange(math.ceil(start / 3) * 3, end + 0.01, 3)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{int(t) % 24:02d}:00" for t in ticks])
    ax.set_xlim(start, end)
    ax.set_xlabel("Time")
    ax.set_ylabel("Caffeine in body (mg)")
    ax.grid(alpha=0.3)


def plot_total(doses, ke, sleep50, sleep30):
    """กราฟ A: คาเฟอีนรวมตามเวลา + เกณฑ์ + เวลาที่นอนได้"""
    t, start, end = _time_axis(doses, sleep50)
    total = [caffeine_at(x, doses, ke) for x in t]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(t, total, color=TOTAL_COLOR, linewidth=2.5, label="Total caffeine")
    ax.axhline(SLEEP_MG, color=OK_COLOR, linestyle="--", label=f"Sleep threshold {SLEEP_MG} mg")
    ax.axhline(STRICT_MG, color=OK_COLOR, linestyle=":", alpha=0.45, label=f"Strict {STRICT_MG} mg")

    if sleep50 is None:
        ax.text(0.98, 0.92, f"Below {SLEEP_MG} mg from the start - sleep any time",
                transform=ax.transAxes, ha="right", color=OK_COLOR, fontweight="bold")
    else:
        label = hours_to_label(sleep50)[:5]
        ax.axvline(sleep50, color=OK_COLOR, linewidth=2)
        ax.plot([sleep50], [SLEEP_MG], "o", color=OK_COLOR)
        ax.annotate(f"Sleep OK {label}", xy=(sleep50, SLEEP_MG), xytext=(-8, 40),
                    textcoords="offset points", ha="right", color=OK_COLOR, fontweight="bold")
    if sleep30 is not None:
        ax.axvline(sleep30, color=OK_COLOR, linestyle=":", alpha=0.45)

    for hour, mg in doses:
        ax.annotate(f"+{mg:.0f}", xy=(hour, caffeine_at(hour, doses, ke)), xytext=(0, 6),
                    textcoords="offset points", ha="center", fontsize=8, color="#555")

    _format_hours(ax, start, end)
    ax.set_ylim(0, max(total) * 1.2)
    ax.set_title("A) Caffeine level over time and when you can sleep")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    return fig


def plot_stacked(doses, ke, labels):
    """กราฟ B: คาเฟอีนที่ค้างจากแต่ละแก้วซ้อนกัน — ผลรวมของชั้น = กราฟ A"""
    t, start, end = _time_axis(doses)
    layers = [[caffeine_at(x, [dose], ke) for x in t] for dose in doses]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.stackplot(t, layers, labels=labels, colors=CUP_COLORS[:len(doses)], alpha=0.6)
    ax.plot(t, np.sum(layers, axis=0), color=TOTAL_COLOR, linewidth=1.5, label="Total = sum of cups")
    ax.axhline(SLEEP_MG, color=OK_COLOR, linestyle="--")

    _format_hours(ax, start, end)
    ax.set_title("B) Caffeine left from each cup (each layer is one term of the sum)")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    return fig

In [ ]:
def show_coffee_menu(coffees=COFFEES):
    print("\nเลือกกาแฟ 1 แก้ว/รายการ (ปริมาณคาเฟอีนโดยประมาณ)")
    for number, (name, volume, low, high) in enumerate(coffees, 1):
        amount = str(low) if low == high else f"{low}–{high}"
        print(f"{number:2}. {name} | {volume} ml | {amount} mg")


def day_label(days_later):
    if days_later == 0:
        return "วันเดียวกัน"
    if days_later == 1:
        return "วันถัดไป"
    return f"อีก {days_later} วัน"


def show_result(weight, age, coffee_name, volume, dose, start_time, pk, alerts):
    print("\n========== ผลการคำนวณ ==========")
    print(f"น้ำหนัก: {weight:.2f} kg")
    print(f"อายุ: {age} ปี")
    print(f"กาแฟ: {coffee_name} ({volume} ml)")
    print(f"คาเฟอีนเริ่มต้นโดยประมาณ: {dose:.2f} mg")
    print(f"เวลาที่ดื่ม: {start_time:%H:%M} น.")

    print("----------------------------------")
    print(f"Vd = {pk['Vd']:.2f} L")
    print(f"C0 = {pk['C0']:.2f} mg/L (ความเข้มข้นเริ่มต้นโดยประมาณ)")
    print(f"CL = {pk['CL']:.3f} L/h")
    print(f"Half-life = {pk['t_half']:.2f} ชั่วโมง")
    print(f"ke = {pk['ke']:.4f} 1/h")
    print("----------------------------------")

    print("เวลาแจ้งเตือนโดยประมาณ (แสดงผลใน Notebook)")
    for a in alerts:
        m = a["minutes"]
        if m == 0:
            print(f"• {a['label']}: ตั้งแต่เวลาดื่ม {start_time:%H:%M} น. (วันเดียวกัน)")
        else:
            print(f"• {a['label']}: {a['end_time']:%H:%M} น. ({day_label(a['days_later'])}) "
                  f"หลังดื่ม {m // 60} ชั่วโมง {m % 60} นาที")

    print("คาเฟอีนไม่ลดถึง 0 mg พอดีตามสมการนี้; ใกล้หมดหมายถึงต่ำกว่า 1 mg")
    print("เป็นค่าประมาณจากแบบจำลองเดิม ไม่ใช่เวลาที่รับรองว่าปลอดคาเฟอีน")
    print("ผลจริงขึ้นอยู่กับแต่ละบุคคลและปริมาณคาเฟอีนจริงในกาแฟ")
    print("==================================")

In [ ]:
MAX_CUPS = 10
# แก้ว 1 แก้ว = (ชื่อ, ชั่วโมงที่ดื่ม, mg ที่ใช้คำนวณ, mg ต่ำสุด, mg สูงสุด)


def validate_cup_count(n):
    if not 1 <= n <= MAX_CUPS:
        raise ValueError(f"จำนวนแก้วต้องอยู่ระหว่าง 1–{MAX_CUPS}")
    return n


def normalize_schedule(cups, bed_hour):
    """เรียงแก้วตามเวลา และถ้าเวลานอน ≤ แก้วสุดท้าย ให้นับเป็นวันถัดไป (+24)"""
    cups = sorted(cups, key=lambda cup: cup[1])
    if bed_hour <= cups[-1][1]:
        bed_hour += 24
    return cups, bed_hour


def to_doses(cups):
    return [(hour, mg) for _, hour, mg, _, _ in cups]


def to_hour(dt):
    return dt.hour + dt.minute / 60


def get_inputs():
    """ถามผู้ใช้จนกว่าจะกรอกถูก คืนค่า (weight, age, cups, bed_hour)"""
    while True:
        try:
            weight = float(input("น้ำหนักตัว (kg): "))
            age = int(input("อายุ (ปี): "))
            if not math.isfinite(weight) or weight <= 0 or age < 0:
                raise ValueError("น้ำหนักหรืออายุไม่ถูกต้อง")

            count = validate_cup_count(int(input(f"วันนี้ดื่มกาแฟกี่แก้ว (1–{MAX_CUPS}): ")))
            show_coffee_menu()

            cups = []
            for i in range(1, count + 1):
                print(f"\n--- แก้วที่ {i} ---")
                choice = int(input(f"เลือกหมายเลขกาแฟ (1–{len(COFFEES)}): "))
                if not 1 <= choice <= len(COFFEES):
                    raise ValueError(f"กรุณาเลือกหมายเลขกาแฟ 1–{len(COFFEES)}")
                name, volume, low, high = COFFEES[choice - 1]

                dose = high
                if low != high:
                    print(f"รายการนี้มีคาเฟอีนประมาณ {low}–{high} mg")
                    estimate = input("ใช้ค่า 1 = ต่ำสุด, 2 = สูงสุด (Enter = สูงสุด): ").strip()
                    if estimate not in ("", "1", "2"):
                        raise ValueError("กรุณาเลือก 1 หรือ 2")
                    dose = low if estimate == "1" else high

                hour = to_hour(convert_time(input(f"เวลาที่ดื่มแก้วที่ {i} (00:00–23:59 วันเดียวกัน): ")))
                cups.append((name, hour, dose, low, high))

            bed_hour = to_hour(convert_time(input("\nเวลาที่ตั้งใจจะนอน (เช่น 23:00): ")))
            cups, bed_hour = normalize_schedule(cups, bed_hour)
            return weight, age, cups, bed_hour

        except ValueError as error:
            print("ข้อมูลไม่ถูกต้อง:", error)
            print("กรุณากรอกข้อมูลใหม่อีกครั้ง\n")

In [ ]:
def main():
    weight, age, coffee_name, volume, dose, start_time = get_inputs()  # 1. รับค่า
    pk = calc_pk(weight, age, dose)                                    # 2. คำนวณ
    alerts = calc_alerts(start_time, dose, pk["ke"])
    show_result(weight, age, coffee_name, volume, dose, start_time, pk, alerts)  # 3. แสดงผล

In [ ]:
# Unit: multi-dose model
ke20 = calc_pk(60, 20, 95)["ke"]
exam = [(8.0, 95), (13.0, 95), (17.0, 126)]
assert abs(caffeine_at(8.0, exam, ke20) - 95) < 1e-9
assert caffeine_at(7.99, exam, ke20) == 0
assert hours_to_label(find_sleep_time(exam, ke20, SLEEP_MG)).startswith("03:01")
assert hours_to_label(find_sleep_time(exam, ke20, STRICT_MG)).startswith("06:37")
assert find_sleep_time([(9.0, 5)], ke20, SLEEP_MG) is None   # Decaf
assert caffeine_at(find_sleep_time(exam, ke20, SLEEP_MG), exam, ke20) <= SLEEP_MG
print("unit: multi-dose model OK")

In [ ]:
# Unit: input helpers
c = lambda h, mg: ("x", h, mg, mg, mg)
cups, bed = normalize_schedule([c(17, 126), c(8, 95), c(13, 95)], 23.0)
assert [x[1] for x in cups] == [8, 13, 17] and bed == 23.0
assert normalize_schedule([c(8, 95), c(17, 126)], 1.0)[1] == 25.0
assert normalize_schedule([c(8, 95), c(0.5, 63)], 23.0)[0][0][1] == 0.5
assert to_doses(cups) == [(8, 95), (13, 95), (17, 126)]
for bad in (0, -1, 11):
    try:
        validate_cup_count(bad)
        raise AssertionError(bad)
    except ValueError:
        pass
assert validate_cup_count(3) == 3
print("unit: input helpers OK")

In [ ]:
# Unit: series
seq = half_life_sequence(96, 4.89, 6)
assert seq[0] == (0, 0.0, 96) and abs(seq[3][2] - 12) < 1e-9
s = multi_day_series(88.0, ke20, 7)
assert abs(s["r"] - math.exp(-24 * ke20)) < 1e-12 and 0.030 < s["r"] < 0.036
assert all(abs(a - b) < 1e-9 for _, a, b in s["rows"])
assert s["rows"][0][1] == 88.0 and s["rows"][-1][1] < s["limit"]
assert abs(s["limit"] - 88.0 / (1 - s["r"])) < 1e-9
ke65 = calc_pk(60, 70, 95)["ke"]
assert multi_day_series(88.0, ke65)["r"] > s["r"]   # ผู้สูงอายุสะสมมากกว่า
print("unit: series OK")

In [ ]:
# Unit: plots
import matplotlib.pyplot as plt
f = plot_total(exam, ke20, find_sleep_time(exam, ke20, SLEEP_MG), find_sleep_time(exam, ke20, STRICT_MG))
ax = f.axes[0]
assert any("03:01" in t.get_text() for t in ax.texts)
lo, hi = ax.get_xlim()
assert hi - lo >= 24
f2 = plot_total([(9.0, 5)], ke20, None, None)
assert f2.axes[0].get_xlim()[1] - f2.axes[0].get_xlim()[0] >= 24
assert any("Below 50 mg" in t.get_text() for t in f2.axes[0].texts)
f3 = plot_stacked(exam, ke20, ["Drip 08:00", "Drip 13:00", "Americano 17:00"])
assert len(f3.axes[0].collections) >= 3
plt.close("all")
print("unit: plots OK")

In [ ]:
main()